In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

In [2]:
PROJECT_PATH = Path.cwd().parent
RAW_PATH = PROJECT_PATH / "01_Raw_Data"
DOCUMENTATION_PATH = PROJECT_PATH / "02_Documentation"
CLEANED_PATH = PROJECT_PATH / "04_Cleaned_Data"
CLEANED_PATH.mkdir(parents=True, exist_ok=True)
print("Project: ", PROJECT_PATH.resolve())

Project:  D:\arc\Python\Python Projects\airline_operations_analytics


In [3]:
RAW_2024_PATH = RAW_PATH / "2024"
print("Raw_2024_Path", RAW_2024_PATH.resolve())

EXTRACT_PATH = RAW_2024_PATH / "January"
print("Extract_Path", EXTRACT_PATH.resolve())

Raw_2024_Path D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024
Extract_Path D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024\January


In [4]:
csv_files = list((RAW_PATH / "2024" / "January").rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No csv file found inside 01_Raw_Data/2024/January"
    )

RAW_FILE = csv_files[0]

df_raw = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

print("Raw File:", RAW_FILE)
print("Raw Shape:", df_raw.shape)

Raw File: d:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024\January\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
Raw Shape: (547271, 110)


In [5]:
raw_rows = len(df_raw)
raw_columns = len(df_raw.columns)
raw_missing_cells = int(df_raw.isna().sum().sum())
raw_duplicate_rows = int(df_raw.duplicated().sum())
raw_empty_columns = int(df_raw.isna().all().sum())

print("Initial Raw Data")
print("Rows: ", raw_rows)
print("Columns: ", raw_columns)
print("Missing Cells: ", raw_missing_cells)
print("Duplicate Rows: ", raw_duplicate_rows)
print("Completely Empty Columns: ", raw_empty_columns)

Initial Raw Data
Rows:  547271
Columns:  110
Missing Cells:  29206067
Duplicate Rows:  0
Completely Empty Columns:  17


In [6]:
df = df_raw.copy()

In [7]:
empty_columns = df.columns[df.isna().all()].to_list()

print("Completely Empty Columns:")
for column in empty_columns:
    print("-", column)

print("\nTotal:", len(empty_columns))

Completely Empty Columns:
- Div4Airport
- Div4AirportID
- Div4AirportSeqID
- Div4WheelsOn
- Div4TotalGTime
- Div4LongestGTime
- Div4WheelsOff
- Div4TailNum
- Div5Airport
- Div5AirportID
- Div5AirportSeqID
- Div5WheelsOn
- Div5TotalGTime
- Div5LongestGTime
- Div5WheelsOff
- Div5TailNum
- Unnamed: 109

Total: 17


In [8]:
df = df.drop(columns=empty_columns)
print("Shape after empty column removal:")
print(df.shape)

Shape after empty column removal:
(547271, 93)


In [9]:
original_columns = df.columns.to_list()

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ","_")
)

print("column names standardized.")

column names standardized.


In [10]:
df["FlightDate"] = pd.to_datetime(
    df["FlightDate"],
    format="%Y-%m-%d",
    errors='coerce'
)

print("Flight Date dtype:", df["FlightDate"].dtype)
print("Invalid dates", df["FlightDate"].isna().sum())
print("Maximum date", df["FlightDate"].max())
print("Minimum date", df["FlightDate"].min())

Flight Date dtype: datetime64[us]
Invalid dates 0
Maximum date 2024-01-31 00:00:00
Minimum date 2024-01-01 00:00:00


In [11]:
numeric_columns = df.select_dtypes(include=np.number).columns
numeric_columns = numeric_columns.to_list()

print("Numeric Column: ", len(numeric_columns))
print("Numeric Column List: ", numeric_columns)

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

Numeric Column:  72
Numeric Column List:  ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'DOT_ID_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'OriginStateFips', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'DestStateFips', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'Cancelled', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime', 'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay', 'DivDistance', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn', 'Div1TotalGTime', 'Div1LongestGTime

In [12]:
categorical_columns = ['Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number',
       'Origin', 'OriginCityName', 'OriginState', 'OriginStateName', 'Dest',
       'DestCityName', 'DestState', 'DestStateName', 'DepTimeBlk',
       'ArrTimeBlk', 'CancellationCode']

for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].astype("category")

In [13]:
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (df.isna().mean().values * 100).round(2)
})

missing_report = missing_report.sort_values(
    "Missing_Percentage",
    ascending=False
)

display(
    missing_report[
        missing_report["Missing_Count"] > 0
    ]
)

,Column,Missing_Count,Missing_Percentage
88,Div3WheelsOn,547270,100.00
89,Div3TotalGTime,547270,100.00
90,Div3LongestGTime,547270,100.00
85,Div3Airport,547270,100.00
86,Div3AirportID,547270,100.00
84,Div2TailNum,547250,100.00
83,Div2WheelsOff,547250,100.00
87,Div3AirportSeqID,547270,100.00
91,Div3WheelsOff,547270,100.00
92,Div3TailNum,547270,100.00


## Cancellation Validation

In [14]:
cancelled_without_code = (
    (df["Cancelled"] == 1) & (df["CancellationCode"].isna())
).sum()

noncancelled_with_code = (
    (df["Cancelled"] == 0) & (df["CancellationCode"].notna())
).sum()

print("Cancelled without cancellation Code:", cancelled_without_code)
print("NonCancelled with cancellation Code:", noncancelled_with_code)

Cancelled without cancellation Code: 0
NonCancelled with cancellation Code: 0


## Delay Logic Validation

In [15]:
dep_delay_logic_1 = (
    (df["DepDel15"] == 1) & (df["DepDelayMinutes"] < 15)
).sum()

dep_delay_logic_2 = (
    (df["DepDel15"] == 0) & (df["DepDelayMinutes"] >= 15)
).sum()

arr_delay_logic_1 = (
    (df["ArrDel15"] == 1) & (df["ArrDelayMinutes"] < 15)
).sum()

arr_delay_logic_2 = (
    (df["ArrDel15"] == 0) & (df["ArrDelayMinutes"] >= 15)
).sum()

print("DepDel15 contradiction #1:", dep_delay_logic_1)
print("DepDel15 contradiction #2:", dep_delay_logic_2)
print("ArrDel15 contradiction #1:", arr_delay_logic_1)
print("ArrDel15 contradiction #2:", arr_delay_logic_2)

DepDel15 contradiction #1: 0
DepDel15 contradiction #2: 0
ArrDel15 contradiction #1: 0
ArrDel15 contradiction #2: 0


## Negative Validation

In [16]:
negative_departure_delay = (
    df["DepDelay"] < 0
).sum()

negative_arrival_delays = (
    df["ArrDelay"] < 0
).sum()

print("Negative Departure Delay: ", negative_departure_delay)
print("Negative Arrival Delay: ", negative_arrival_delays)

Negative Departure Delay:  292317
Negative Arrival Delay:  306999


## Diversion Logic

In [17]:
diverted_flights = (
    df["Diverted"] ==1
).sum()

print("Diverted Flights: ", diverted_flights)

print("Diverted Flights with ArrDelay populated:",
      (
          (df["Diverted"] == 1) & (df["ArrDelay"].notna())
      ).sum()
    )

print("Diverted Flights with ArrDelay Null:",
      (
          (df["Diverted"] == 1) & (df["ArrDelay"].isna())
      ).sum()
    )

Diverted Flights:  1512
Diverted Flights with ArrDelay populated: 0
Diverted Flights with ArrDelay Null: 1512


## Time Field Validation

In [18]:
time_columns = [
    "CRSDepTime",
    "DepTime",
    "CRSArrTime",
    "ArrTime",
    "WheelsOff",
    "WheelsOn"
]

for column in time_columns:
    if column in df.columns:
        invalid = (
            df[column].notna() &
            ~df[column].between(0, 2400)
        ).sum()
        print(f"{column}: invalid values = {invalid}")

CRSDepTime: invalid values = 0
DepTime: invalid values = 0
CRSArrTime: invalid values = 0
ArrTime: invalid values = 0
WheelsOff: invalid values = 0
WheelsOn: invalid values = 0


## Range Validation

In [19]:
range_checks = {
    "Quarter outside 1-4":
    (~df["Quarter"].between(1,4)).sum(),

    "Month outside 1-12":
    (~df["Month"].between(1,12)).sum(),

    "DayOfWeek outside 1-7":
    (~df["DayOfWeek"].between(1,7)).sum(),

    "Cancelled outside 0/1":
    (~df["Cancelled"].isin([0,1])).sum(),

    "Diverted outside 0/1":
    (~df["Diverted"].isin([0,1])).sum(),

    "Distance <=0":
    (df["Distance"]<=0).sum(),

    "Flights <=0":
    (df["Flights"] <=0).sum()

}

for check, count in range_checks.items():
    print(f"{check}: {count}")

Quarter outside 1-4: 0
Month outside 1-12: 0
DayOfWeek outside 1-7: 0
Cancelled outside 0/1: 0
Diverted outside 0/1: 0
Distance <=0: 0
Flights <=0: 0


## Categorical Quality

In [20]:
print("Reporting Airline")
print(
    df["Reporting_Airline"].value_counts()
)

print("Cancellation Code: ")
print(
    df["CancellationCode"].value_counts(dropna= False)
)

Reporting Airline
Reporting_Airline
WN    115389
AA     77346
DL     74384
UA     58855
OO     56814
YX     22914
MQ     20750
NK     20415
B6     19580
AS     17775
9E     16972
OH     16526
F9     14379
G4      8596
HA      6576
Name: count, dtype: int64
Cancellation Code: 
CancellationCode
NaN    526882
B       12085
A        7736
C         568
Name: count, dtype: int64


## Duplicate Validation

In [22]:
duplicate_count = df.duplicated().sum()
print(
    "Duplicate rows after cleaning: ",
    duplicate_count
)

Duplicate rows after cleaning:  0


## Final Empty Column Check

In [23]:
remaining_empty_columns = df.columns[df.isna().all()].tolist()

print("Remaining Completely Empty Columns: ", len(remaining_empty_columns))

print(remaining_empty_columns)

Remaining Completely Empty Columns:  0
[]


## Final Data-Type Audit

In [24]:
dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage":(
        df.isna().mean().values * 100
    ).round(2),
    "Unique Values": df.nunique(
        dropna = True
    ).values
})

display(dtype_report)

,Column,Data_Type,Missing_Count,Missing_Percentage,Unique Values
0,Year,int64,0,0.0,1
1,Quarter,int64,0,0.0,1
2,Month,int64,0,0.0,1
3,DayofMonth,int64,0,0.0,31
4,DayOfWeek,int64,0,0.0,7
...,...,...,...,...,...
88,Div3WheelsOn,float64,547270,100.0,1
89,Div3TotalGTime,float64,547270,100.0,1
90,Div3LongestGTime,float64,547270,100.0,1
91,Div3WheelsOff,float64,547270,100.0,1


In [25]:
dtype_report.to_excel(
    DOCUMENTATION_PATH / "Post_Cleaning_Data_Profile.xlsx",
    index=False
)

## Cleaning Reconciliation

In [28]:
cleaned_rows = len(df)
cleaned_columns = len(df.columns)

reconciliation = pd.DataFrame({
    "Metric":[
        "Raw Rows",
        "Cleaned Rows",
        "Rows Removed",
        "Raw Columns",
        "Cleaned Columns",
        "Columns Removed",
        "Raw Duplicate Rows",
        "Cleaned Duplicate Rows",
        "Raw Missing Cells",
        "Cleaned Missing Cells"
    ],
    "Value":[
        raw_rows,
        cleaned_rows,
        raw_rows - cleaned_rows,
        raw_columns,
        cleaned_columns,
        raw_columns - cleaned_columns,
        raw_duplicate_rows,
        duplicate_count,
        raw_missing_cells,
        int(df.isna().sum().sum())

    ]
})

display(reconciliation)

,Metric,Value
0,Raw Rows,547271
1,Cleaned Rows,547271
2,Rows Removed,0
3,Raw Columns,110
4,Cleaned Columns,93
5,Columns Removed,17
6,Raw Duplicate Rows,0
7,Cleaned Duplicate Rows,0
8,Raw Missing Cells,29206067
9,Cleaned Missing Cells,19902460


## Save Clean Data

In [30]:
cleaned_file = (
    CLEANED_PATH / "Airline_Flights_January_2024_Cleaned.csv"
)

df.to_csv(cleaned_file, index=False)

print("Cleaned dataset saved successfully.")
print(cleaned_file.resolve())

Cleaned dataset saved successfully.
D:\arc\Python\Python Projects\airline_operations_analytics\04_Cleaned_Data\Airline_Flights_January_2024_Cleaned.csv


## Cleaned Audit Log

In [33]:
cleaning_audit = pd.DataFrame({
    "Step":[
        "Raw dataset preserved",
        "Baseline Captured",
        "Completely empty columns removed"
    ],
    "Staus":[
        "Completed",
        "Completed",
        "Completed"
    ]

})

display(cleaning_audit)

,Step,Staus
0,Raw dataset preserved,Completed
1,Baseline Captured,Completed
2,Completely empty columns removed,Completed


In [34]:
cleaning_audit.to_excel(
    DOCUMENTATION_PATH / "Airline_Cleaning_Audit.xlsx",
    index=False
)